# EvalNova — fine-tuning TrOCR on handwritten C code

**Run this on Google Colab with a GPU.** Runtime → Change runtime type → T4 GPU.

## Why fine-tune at all

`trocr-small-handwritten` was trained on the IAM corpus: British and European
adults writing everyday English prose on clean paper. Our benchmark on synthetic
handwritten C measured **27% character error**, and the failures are not blurry
images — they are symbols the model has never been taught:

| Written | Read as |
|---|---|
| `#include <stdio.h>` | `tinclude estdio-ho-` |
| `int main()` | `int mainly` |
| `flag = 1;` | `flag - 1 ;` |
| `scanf("%d", &n);` | `scant ( " rod " , en ) ;` |

Clean pages scored 27.4% and badly degraded pages 25.1% — image quality barely
matters. The model is the bottleneck, so the model is what we change.

## What this notebook does

1. Loads line crops exported by `scripts/export_line_crops.py`
2. Fine-tunes `trocr-small-handwritten` on them
3. Reports character error rate on a held-out split after every epoch
4. Saves a model directory you drop straight back into EvalNova

**Run order:** synthetic data first (teaches the symbols), then re-run including
the real scripts when they arrive (teaches the handwriting).

## 1 · Check the GPU

If this prints `cpu`, stop and switch the runtime — training on CPU takes days.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("NO GPU — Runtime > Change runtime type > T4 GPU, then run again.")

In [ ]:
# transformers is pinned below 5.0: the 5.x line removed the slow-to-fast
# tokenizer conversion the TrOCR checkpoints depend on, and loading fails there.
!pip install -q "transformers<5.0" jiwer sentencepiece protobuf

## 2 · Upload the training data

On your laptop:

```bash
python scripts/generate_code_dataset.py --pages 400 --out data/synthetic
python scripts/export_line_crops.py --dataset data/synthetic --out data/train
```

Then zip `data/train` and upload it — or put the zip in Drive and mount it, which
is less painful for anything over a few hundred megabytes.

In [ ]:
from pathlib import Path

USE_DRIVE = False  # True if you put train.zip in Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !cp "/content/drive/MyDrive/train.zip" /content/train.zip
else:
    from google.colab import files
    files.upload()  # choose train.zip

!unzip -q -o train.zip -d /content/data
DATA = Path("/content/data/train")
if not (DATA / "train.jsonl").exists():
    DATA = next(p.parent for p in Path("/content/data").rglob("train.jsonl"))
print("data at:", DATA)
print(open(DATA / "summary.json").read())

## 3 · The dataset

One example is a cropped line image and the text written on it. The processor
turns the image into the fixed-size tensor the vision encoder expects and the text
into token ids. Padding tokens are set to `-100` so the loss ignores them —
without that the model is rewarded for predicting padding.

In [ ]:
import json
import random

from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

BASE_MODEL = "microsoft/trocr-small-handwritten"
MAX_TARGET_LENGTH = 96
MAX_TRIVIAL_SHARE = 0.05   # at most 5% of training lines may be 1-2 characters

processor = TrOCRProcessor.from_pretrained(BASE_MODEL)


def cap_trivial(rows, share=MAX_TRIVIAL_SHARE, seed=0):
    """Stop one-character lines from teaching the model to say nothing.

    Code is full of lines holding a single brace, so they dominate the data:
    14% of our crops are '{' or '}'. Trained on that, the model learns brevity
    as a habit and starts dropping semicolons, operators and digits from real
    lines - which is exactly what the first run did. Keep a few, not a seventh.
    """
    trivial = [r for r in rows if len(r["text"]) <= 2]
    rest = [r for r in rows if len(r["text"]) > 2]
    keep = int(len(rest) * share / max(1 - share, 1e-6))
    random.Random(seed).shuffle(trivial)
    kept = trivial[:keep]
    print(f"trivial lines: {len(trivial)} found, {len(kept)} kept")
    out = rest + kept
    random.Random(seed).shuffle(out)
    return out


class LineDataset(Dataset):
    def __init__(self, root: Path, split: str, balance: bool = False):
        self.root = root
        with open(root / f"{split}.jsonl", encoding="utf-8") as fh:
            rows = [json.loads(line) for line in fh]
        self.rows = cap_trivial(rows) if balance else rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        image = Image.open(self.root / row["image"]).convert("RGB")
        pixel_values = processor(images=image, return_tensors="pt").pixel_values[0]
        labels = processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
        ).input_ids
        labels = [t if t != processor.tokenizer.pad_token_id else -100 for t in labels]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}


# Balance the training split only. The validation split stays as it is, so the
# before/after numbers are measured against the same thing as the first run.
train_ds = LineDataset(DATA, "train", balance=True)
val_ds = LineDataset(DATA, "val")
print(f"train {len(train_ds)} lines | val {len(val_ds)} lines")
print("example:", train_ds.rows[0]["text"])


## 4 · The model

The decoder needs its special tokens set explicitly — the base checkpoint ships
without them wired up, and generation produces nonsense if they are left unset.

`num_beams=1` keeps greedy decoding, matching how EvalNova runs it in production:
evaluating with beams and serving greedily would flatter the numbers.

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL).to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.vocab_size = model.config.decoder.vocab_size

model.generation_config.max_new_tokens = MAX_TARGET_LENGTH
model.generation_config.num_beams = 1
model.generation_config.early_stopping = False

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

## 5 · Train

Character error rate is computed on the validation split after every epoch. Watch
that number, not the loss — loss falling while CER stalls means the model is
memorising the fonts rather than learning the letters.

On a T4, roughly 6,000 synthetic lines take about 20–30 minutes for 4 epochs.

In [ ]:
import jiwer
import numpy as np
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments


def compute_metrics(pred):
    """Character and word error rate on the validation split.

    Both the predictions and the labels need cleaning before they can be decoded.
    The trainer pads each batch to the longest row using -100, and the tokenizer
    raises OverflowError on a negative id, so every out-of-range value becomes the
    pad token first.
    """
    pred_ids = pred.predictions
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    pred_ids = np.asarray(pred_ids)
    if pred_ids.ndim == 3:          # logits rather than ids
        pred_ids = pred_ids.argmax(-1)

    pad_id = processor.tokenizer.pad_token_id
    vocab = len(processor.tokenizer)
    pred_ids = np.where((pred_ids < 0) | (pred_ids >= vocab), pad_id, pred_ids)
    label_ids = np.where(pred.label_ids != -100, pred.label_ids, pad_id)

    references = processor.batch_decode(label_ids, skip_special_tokens=True)
    predictions = processor.batch_decode(pred_ids, skip_special_tokens=True)
    pairs = [(r, p) for r, p in zip(references, predictions) if r.strip()]
    if not pairs:
        return {"cer": 1.0, "wer": 1.0}
    refs, hyps = zip(*pairs)
    return {
        "cer": float(jiwer.cer(list(refs), list(hyps))),
        "wer": float(jiwer.wer(list(refs), list(hyps))),
    }


args = Seq2SeqTrainingArguments(
    output_dir="/content/trocr-code",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,   # 5e-5 drifted: CER rose while WER fell
    num_train_epochs=2,
    warmup_ratio=0.1,
    fp16=(device == "cuda"),
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    logging_steps=50,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=lambda batch: {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    },
)

baseline = trainer.evaluate()
print(f"BEFORE fine-tuning:  CER {baseline['eval_cer']:.1%}  WER {baseline['eval_wer']:.1%}")

In [ ]:
trainer.train()

final = trainer.evaluate()
print(f"\nBEFORE: CER {baseline['eval_cer']:.1%}   AFTER: CER {final['eval_cer']:.1%}")
print(f"BEFORE: WER {baseline['eval_wer']:.1%}   AFTER: WER {final['eval_wer']:.1%}")

## 6 · Look at what it actually predicts

A number can improve while the output is still unusable. Read a few lines.

In [ ]:
model.eval()
for row in val_ds.rows[:12]:
    image = Image.open(DATA / row["image"]).convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
    with torch.inference_mode():
        generated = model.generate(pixel_values, max_new_tokens=MAX_TARGET_LENGTH)
    predicted = processor.batch_decode(generated, skip_special_tokens=True)[0]
    mark = "  " if predicted.strip() == row["text"].strip() else "<<"
    print(f"{mark} TRUE: {row['text']}")
    print(f"   READ: {predicted}")

## 7 · Save and download

The processor is saved alongside the weights. EvalNova loads both from the same
directory, so they must travel together.

In [ ]:
OUT = "/content/trocr-code-final"
model.save_pretrained(OUT)
processor.save_pretrained(OUT)

!cd /content && zip -qr trocr-code-final.zip trocr-code-final
!du -h /content/trocr-code-final.zip

from google.colab import files
files.download("/content/trocr-code-final.zip")

## 8 · Put it back into EvalNova

Unzip into `models/trocr-code-final` in the repository, then point the config at
that folder — the provider loads a local directory exactly like a Hugging Face
name, so **no code changes are needed**:

```
EVALNOVA_OCR_MODEL_NAME=C:\...\evalnova-core\models	rocr-code-final
```

Then measure on **`data/synthetic_test`** — the 40-page held-out set the baseline
used. It was generated with a different seed from the training data, so the model
has never seen these pages. Using any other set makes the comparison meaningless.

```bash
python scripts/benchmark.py --dataset data/synthetic_test --tag after-finetune
```

**The baseline to beat, measured on that exact set:**

| | CER | WER |
|---|---|---|
| overall | 30.4% | 85.6% |
| code pages | 31.3% | 87.2% |
| prose pages | 12.0% | 55.7% |

**Beware of one trap.** The validation split inside this notebook is synthetic and
drawn from the same fonts as the training split, so a low CER there mostly proves
the model learned those fonts. The number that counts is the benchmark on **real
scanned scripts**, which no amount of synthetic data can stand in for. Expect the
real figure to be substantially worse, and report that one.
